In [15]:
import ifcopenshell
import ifcopenshell.geom
import ifcopenshell.util.shape

import numpy as np
import plotly.graph_objects as go

In [10]:
def get_line_geometry(model_name, ifc_type):
    # Load IFC and extract elements
    model = ifcopenshell.open(model_name)
    settings = ifcopenshell.geom.settings()
    elements = model.by_type(ifc_type)
    # Initialize arrays
    element_centers, element_starts, element_ends = [], [], []

    for element in elements:
        try:
            shape = ifcopenshell.geom.create_shape(settings, element)
            # Global vertex coordinates
            vertices = np.array(ifcopenshell.util.shape.get_shape_vertices(shape, shape.geometry))
            # Find main beam direction using PCA
            center = vertices.mean(axis=0)
            U, S, v = np.linalg.svd(vertices - center)
            direction = v[0]
            projections = (vertices - center) @ direction
            # Find endpoints
            start = center + projections.min() * direction
            end = center + projections.max() * direction
            # Store points
            element_centers.append(center)
            element_starts.append(start)
            element_ends.append(end)
        except Exception as e:
            print(f"Could not process element {element.Name}: {e}")

    # Convert to numpy arrays
    element_centers, element_starts, element_ends = np.array(element_centers), np.array(element_starts), np.array(element_ends)
    return element_centers, element_starts, element_ends

In [11]:
model_name = "26_02_D_STR.ifc" # 26-01-D-STR.ifc
ifc_type = "IfcBeam" # IfcColumn
beam_centers, beam_starts, beam_ends = get_line_geometry(model_name, ifc_type)
print(f"Beam 1:\nBeam center: {beam_centers[0]}\nBeam start: {beam_starts[0]}\nBeam end: {beam_ends[0]}")

Beam 1:
Beam center: [ 13.5180205  -26.92686253   6.7       ]
Beam start: [ 15.04810811 -27.39465726   6.7       ]
Beam end: [ 11.9879329 -26.4590678   6.7      ]


In [12]:
model_name = "26_02_D_STR.ifc" # 26-01-D-STR.ifc
ifc_type = "IfcColumn" # IfcBeam
column_centers, column_starts, column_ends = get_line_geometry(model_name, ifc_type)
print(f"column 1:\nColumn center: {column_centers[0]}\nColumn start: {column_starts[0]}\nColumn end: {column_ends[0]}")

column 1:
Column center: [ -8.75895094 -39.59420921  -1.875     ]
Column start: [ -8.75895094 -39.59420921  -0.14      ]
Column end: [ -8.75895094 -39.59420921  -3.61      ]


In [ ]:
def get_grid_lines(model_name):
    model = ifcopenshell.open(model_name)
    local_coords = ifcopenshell.util.geolocation.local2global()

    return

In [ ]:
def transform_point(matrix, point_coords):
    # Transforms a 2D or 3D local point using a 4x4 placement matrix.
    # Ensure point is 3D (IFC grid curves are often 2D)
    x = point_coords[0]
    y = point_coords[1]
    z = point_coords[2] if len(point_coords) > 2 else 0.0

    local_pt = np.array([x, y, z, 1.0])
    global_pt = matrix @ local_pt
    return tuple(round(float(coord), 4) for coord in global_pt[:3])


def get_curve_endpoints(axis_curve):
    # Extracts the local start and end points from common IFC curve types.
    if axis_curve.is_a("IfcPolyline"):
        points = axis_curve.Points
        return points[0].Coordinates, points[-1].Coordinates

    elif axis_curve.is_a("IfcTrimmedCurve"):
        # Handle trimmed lines/curves
        basis = axis_curve.BasisCurve
        if basis.is_a("IfcLine"):
            origin = np.array(basis.Pnt.Coordinates)
            direction = np.array(basis.Dir.OrientationRatios)
            magnitude = basis.Dir.Magnitude
            # Normalize direction vector
            unit_dir = direction / np.linalg.norm(direction)

            # Trim1 and Trim2 usually contain parameter values (IfcParameterValue)
            t1 = axis_curve.Trim1[0].wrappedValue
            t2 = axis_curve.Trim2[0].wrappedValue

            p1 = origin + unit_dir * (t1 * magnitude)
            p2 = origin + unit_dir * (t2 * magnitude)
            return tuple(p1), tuple(p2)

    raise NotImplementedError(f"Curve type {axis_curve.is_a()} is not yet handled.")


# Load your IFC file
ifc_file = ifcopenshell.open("26_02_D_STR.ifc")
grids = ifc_file.by_type("IfcGrid")
# Initialize output arrays
global_start = []
global_end = []
tag = []

# Iterate through all grids and their axes
for grid in grids:

    # Get the 4x4 transformation matrix for the entire IfcGrid
    grid_matrix = ifcopenshell.util.placement.get_local_placement(grid.ObjectPlacement)

    # Combine U, V, and W axes (W is optional in IFC, so default to empty tuple)
    all_axes = (("U", grid.UAxes or ()), ("V", grid.VAxes or ()), ("W", grid.WAxes or ()),)

    for direction_label, axes in all_axes:
        for axis in axes:
            tag.append(axis.AxisTag or "?")
            try:
                local_start, local_end = get_curve_endpoints(axis.AxisCurve)
                global_start.append(transform_point(grid_matrix, local_start))
                global_end.append(transform_point(grid_matrix, local_end))

                print(f"[{direction_label}-Axis] Tag: {tag[-1]} | Start: {global_start[-1]} | End: {global_end[-1]}")
            except Exception as e:
                print(f"[{direction_label}-Axis] Tag: {tag[-1]} | Error: {e}")



Grid: Default Grid
  [U-Axis] Tag: 1 | Start: (741.7965, 2426.3071, -3070.0) | End: (-13558.5469, -44348.0087, -3070.0)
  [U-Axis] Tag: 2 | Start: (3801.9717, 1490.7177, -3070.0) | End: (-10498.3717, -45283.5982, -3070.0)
  [U-Axis] Tag: 3 | Start: (6862.147, 555.1282, -3070.0) | End: (-7438.1965, -46219.1876, -3070.0)
  [U-Axis] Tag: 4 | Start: (9922.3222, -380.4612, -3070.0) | End: (-4378.0213, -47154.7771, -3070.0)
  [U-Axis] Tag: 5 | Start: (12982.4974, -1316.0507, -3070.0) | End: (-1317.846, -48090.3665, -3070.0)
  [U-Axis] Tag: 6 | Start: (16042.6726, -2251.6402, -3070.0) | End: (1742.3292, -49025.956, -3070.0)
  [U-Axis] Tag: 7 | Start: (19102.8478, -3187.2296, -3070.0) | End: (4802.5044, -49961.5454, -3070.0)
  [U-Axis] Tag: 8 | Start: (22163.0231, -4122.8191, -3070.0) | End: (7862.6796, -50897.1349, -3070.0)
  [U-Axis] Tag: 9 | Start: (25223.1983, -5058.4085, -3070.0) | End: (10922.8548, -51832.7243, -3070.0)
  [U-Axis] Tag: 10 | Start: (28283.3735, -5993.998, -3070.0) | End: 

In [22]:
# Create interactive 3D figure
fig = go.Figure()

# Plot all beams as lines
for start, end in zip(global_start, global_end):

    fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]],
        y=[start[1], end[1]],
        z=[start[2], end[2]],
        mode="lines", line=dict(color="red", width=3), showlegend=False))

# Layout settings
fig.update_layout(
    title="Interactive 3D IFC Beam Model",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"), width=1200, height=900)

# Display interactive plot
fig.show()